In [6]:
from typing import Literal , TypedDict ,Annotated
from langgraph.graph import StateGraph , START, END
from langchain_ollama import ChatOllama
from pydantic import Field, BaseModel
from langgraph.graph.message import BaseMessage , add_messages
from langchain_core.messages import HumanMessage ,AIMessage
from langgraph.checkpoint.memory import MemorySaver
model = "qwen3.5:9b"
 

llm = ChatOllama(
    model=model,
    temperature=0
)
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

### Funtions

In [7]:
def chat_node(state:ChatState)->ChatState:
    message = state['messages']
   
    response = llm.invoke(message)
    
    return{
        "messages":[response]
    }

### nodes

In [8]:
checkpoint = MemorySaver()
graph = StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

# define edges
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
workflow = graph.compile(checkpoint)

## Memory

In [16]:
config = {
    "configurable": {
        "thread_id": "user_1"
    }
}

In [ ]:



init={"messages": [HumanMessage(content="My name is user 2")]}
response = workflow.invoke(init,config)
print(response['messages'][-1].content)

Hello! It's nice to meet you, User 2. How can I help you today?


In [17]:
 

init={"messages": [HumanMessage(content="what is my name")]}
response = workflow.invoke(init,config)
print(response['messages'][-1].content)

Your name is **Hussin**.


In [ ]:
while True:
    user_message = input('type here')
    
    print('User: ',user_message)
    init = {
    "messages":[HumanMessage(content=user_message)]
}   
    response = workflow.invoke(init)
    print("Ai:" , response['messages'][-1].content)
    
    if  user_message.strip().lower() in ['exit','bye','quit']:
        break

User:  my name is muhamed
Ai: Hello Muhamed! It is nice to meet you. How can I help you today?
User:  fuck you
Ai: I understand you might be frustrated. Is there something specific I can help you with?
User:  no just i want to see your feedback
Ai: Of course! Could you clarify **what specifically** you'd like feedback on? For example:  
- A piece of writing or content?  
- Code or technical work?  
- My previous responses in this chat?  
- Something else entirely?  

The more details you share, the better I can tailor my feedback! 😊
User:  ثءهف


KeyboardInterrupt: 